In [1]:
import pandas as pd
import numpy as np
import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# ==========================================
# LOAD SAVED MODELS
# ==========================================

isolation_forest = joblib.load(
    "../models/isolation_forest.pkl"
)

lof = joblib.load(
    "../models/lof.pkl"
)

scaler = joblib.load(
    "../models/robust_scaler.pkl"
)

best_threshold = joblib.load(
    "../models/isolation_forest_threshold.pkl"
)

print("Models loaded successfully!")
print("--------------------------------")
print("Isolation Forest :", type(isolation_forest).__name__)
print("LOF              :", type(lof).__name__)
print("Scaler           :", type(scaler).__name__)
print(f"Threshold        : {best_threshold:.6f}")

Models loaded successfully!
--------------------------------
Isolation Forest : IsolationForest
LOF              : LocalOutlierFactor
Scaler           : RobustScaler
Threshold        : 0.115433


In [7]:
# ==========================================
# CORRECTED REAL-TIME PREDICTION FUNCTION
# ==========================================

def predict_transaction(transaction_data):

    # Convert input into DataFrame
    transaction_df = pd.DataFrame([transaction_data])

    # Scale Amount using Amount scaler
    transaction_df["Amount"] = amount_scaler.transform(
        transaction_df[["Amount"]]
    )

    # Scale Time using Time scaler
    transaction_df["Time"] = time_scaler.transform(
        transaction_df[["Time"]]
    )

    # Make sure feature order is exactly the same
    feature_order = [
        "Time",
        "V1", "V2", "V3", "V4", "V5", "V6", "V7",
        "V8", "V9", "V10", "V11", "V12", "V13", "V14",
        "V15", "V16", "V17", "V18", "V19", "V20", "V21",
        "V22", "V23", "V24", "V25", "V26", "V27", "V28",
        "Amount"
    ]

    transaction_df = transaction_df[feature_order]

    # Calculate anomaly score
    anomaly_score = -isolation_forest.decision_function(
        transaction_df
    )[0]

    # Apply selected threshold
    if anomaly_score >= best_threshold:
        prediction = "ANOMALY / POSSIBLE FRAUD"
    else:
        prediction = "NORMAL"

    return prediction, anomaly_score


print("Corrected prediction function created successfully!")

Corrected prediction function created successfully!


In [5]:
# ==========================================
# CREATE CORRECT SCALERS
# ==========================================

from sklearn.preprocessing import RobustScaler
import pandas as pd
import joblib

# Load original dataset
df_original = pd.read_csv("../dataset/creditcard.csv")

# Create separate scalers
amount_scaler = RobustScaler()
time_scaler = RobustScaler()

# Fit scalers using original training data
amount_scaler.fit(df_original[["Amount"]])
time_scaler.fit(df_original[["Time"]])

# Save them
joblib.dump(
    amount_scaler,
    "../models/amount_scaler.pkl"
)

joblib.dump(
    time_scaler,
    "../models/time_scaler.pkl"
)

print("Correct scalers created and saved!")
print("----------------------------------")
print("1. amount_scaler.pkl")
print("2. time_scaler.pkl")

Correct scalers created and saved!
----------------------------------
1. amount_scaler.pkl
2. time_scaler.pkl


In [8]:
# ==========================================
# TEST WITH A REAL TRANSACTION
# ==========================================

# Load original dataset
df_test = pd.read_csv("../dataset/creditcard.csv")

# Select one transaction
sample_transaction = df_test.drop("Class", axis=1).iloc[0].to_dict()

# Actual class from dataset
actual_class = df_test["Class"].iloc[0]

# Make prediction
prediction, anomaly_score = predict_transaction(sample_transaction)

print("=" * 50)
print("REAL-TIME TRANSACTION TEST")
print("=" * 50)

print(f"Actual Class   : {actual_class}")
print(f"Prediction     : {prediction}")
print(f"Anomaly Score  : {anomaly_score:.6f}")
print(f"Threshold      : {best_threshold:.6f}")

REAL-TIME TRANSACTION TEST
Actual Class   : 0
Prediction     : NORMAL
Anomaly Score  : -0.090602
Threshold      : 0.115433


In [9]:
# ==========================================
# TEST WITH A REAL FRAUD TRANSACTION
# ==========================================

# Select the first actual fraud transaction
fraud_transaction = df_test[df_test["Class"] == 1].iloc[0]

# Remove Class before prediction
fraud_input = fraud_transaction.drop("Class").to_dict()

# Predict
fraud_prediction, fraud_anomaly_score = predict_transaction(
    fraud_input
)

print("=" * 50)
print("REAL-TIME FRAUD TRANSACTION TEST")
print("=" * 50)

print(f"Actual Class   : {int(fraud_transaction['Class'])}")
print(f"Prediction     : {fraud_prediction}")
print(f"Anomaly Score  : {fraud_anomaly_score:.6f}")
print(f"Threshold      : {best_threshold:.6f}")

REAL-TIME FRAUD TRANSACTION TEST
Actual Class   : 1
Prediction     : NORMAL
Anomaly Score  : 0.019148
Threshold      : 0.115433


In [10]:
# ==========================================
# TEST 100 REAL FRAUD TRANSACTIONS
# ==========================================

fraud_samples = df_test[df_test["Class"] == 1].head(100)

detected = 0
missed = 0

for _, fraud_transaction in fraud_samples.iterrows():

    fraud_input = fraud_transaction.drop("Class").to_dict()

    prediction, anomaly_score = predict_transaction(fraud_input)

    if prediction == "ANOMALY / POSSIBLE FRAUD":
        detected += 1
    else:
        missed += 1

total_tested = len(fraud_samples)
detection_rate = (detected / total_tested) * 100

print("=" * 50)
print("100 FRAUD TRANSACTIONS TEST")
print("=" * 50)

print(f"Total Fraud Tested : {total_tested}")
print(f"Fraud Detected     : {detected}")
print(f"Fraud Missed       : {missed}")
print(f"Detection Rate     : {detection_rate:.2f}%")
print("=" * 50)

100 FRAUD TRANSACTIONS TEST
Total Fraud Tested : 100
Fraud Detected     : 55
Fraud Missed       : 45
Detection Rate     : 55.00%


In [12]:
# ==========================================
# PREPARE DATA FOR THRESHOLD COMPARISON
# ==========================================

from sklearn.preprocessing import RobustScaler

# Load original dataset
df_model = pd.read_csv("../dataset/creditcard.csv")

# Create separate scalers
amount_scaler_model = RobustScaler()
time_scaler_model = RobustScaler()

# Scale Amount and Time
df_model["Amount"] = amount_scaler_model.fit_transform(
    df_model[["Amount"]]
)

df_model["Time"] = time_scaler_model.fit_transform(
    df_model[["Time"]]
)

# Separate features and target
X = df_model.drop("Class", axis=1)
y = df_model["Class"]

print("Data prepared successfully!")
print("X shape:", X.shape)
print("y shape:", y.shape)

Data prepared successfully!
X shape: (284807, 30)
y shape: (284807,)


In [13]:
# ==========================================
# THRESHOLD COMPARISON
# ==========================================

from sklearn.metrics import precision_score, recall_score, f1_score

# Calculate anomaly scores for all transactions
all_scores = -isolation_forest.decision_function(X)

# Thresholds to compare
test_thresholds = [0.115433, 0.100, 0.080, 0.060, 0.040, 0.020, 0.010]

threshold_results = []

for threshold in test_thresholds:

    predictions = (all_scores >= threshold).astype(int)

    precision = precision_score(y, predictions, zero_division=0)
    recall = recall_score(y, predictions, zero_division=0)
    f1 = f1_score(y, predictions, zero_division=0)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

threshold_comparison = pd.DataFrame(threshold_results)

print("=" * 70)
print("THRESHOLD COMPARISON")
print("=" * 70)

print(threshold_comparison.to_string(index=False))

THRESHOLD COMPARISON
 Threshold  Precision   Recall  F1 Score
  0.115433   0.137544 0.398374  0.204486
  0.100000   0.108748 0.457317  0.175713
  0.080000   0.089689 0.556911  0.154497
  0.060000   0.073493 0.656504  0.132187
  0.040000   0.057411 0.709350  0.106224
  0.020000   0.046232 0.780488  0.087293
  0.010000   0.041123 0.813008  0.078286
